# 03 — ERA5 variables (Medellín)

Unlike Bogotá (which uses SISAIRE station gauges for wind/temperature/RH/pressure,
reserving ERA5 for BLH/radiation/precipitation only), Medellín has no comparably good
local meteorology in what we have locally (`DATOS_MEDELLIN` is only 3 unidentified
stations at minute resolution). So ERA5 supplies **all** of BLH, radiation,
precipitation, wind speed/direction, temperature, RH and pressure here.

Downloads: `boundary_layer_height`, `surface_solar_radiation_downwards`,
`total_precipitation`, `10m_u_component_of_wind`, `10m_v_component_of_wind`,
`2m_temperature`, `2m_dewpoint_temperature` (for RH), `surface_pressure`, 2015-2022
(matching the pollutant `Hourmean` data's coverage).


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import pandas as pd
from common.external_data import download_era5_hourly, load_era5_daily, MEDELLIN_ERA5_AREA

OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "medellin"
RAW_ERA5_DIR = PROJECT_ROOT / "Data" / "external" / "era5_medellin"

start_year, end_year = 2015, 2022
variables = [
    "boundary_layer_height", "surface_solar_radiation_downwards", "total_precipitation",
    "10m_u_component_of_wind", "10m_v_component_of_wind",
    "2m_temperature", "2m_dewpoint_temperature", "surface_pressure",
]
era5_paths = download_era5_hourly(
    start_year, end_year, RAW_ERA5_DIR, area=MEDELLIN_ERA5_AREA, variables=variables,
)
print(f"{len(era5_paths)} files")


[download_era5_hourly] blh 2015 already downloaded, skipping
[download_era5_hourly] blh 2016 already downloaded, skipping
[download_era5_hourly] blh 2017 already downloaded, skipping
[download_era5_hourly] blh 2018 already downloaded, skipping
[download_era5_hourly] blh 2019 already downloaded, skipping
[download_era5_hourly] blh 2020 already downloaded, skipping
[download_era5_hourly] blh 2021 already downloaded, skipping
[download_era5_hourly] blh 2022 already downloaded, skipping
[download_era5_hourly] ssrd 2015 already downloaded, skipping
[download_era5_hourly] ssrd 2016 already downloaded, skipping
[download_era5_hourly] ssrd 2017 already downloaded, skipping
[download_era5_hourly] ssrd 2018 already downloaded, skipping
[download_era5_hourly] ssrd 2019 already downloaded, skipping
[download_era5_hourly] ssrd 2020 already downloaded, skipping
[download_era5_hourly] ssrd 2021 already downloaded, skipping
[download_era5_hourly] ssrd 2022 already downloaded, skipping
[download_era5_h

In [2]:
era5_daily = load_era5_daily(era5_paths)
print(era5_daily.shape)
print(era5_daily.columns.tolist())
era5_daily.describe()


(2923, 13)
['date', 'blh_mean', 'blh_min', 'blh_max', 'blh_morning_mean', 'radiation_mean', 'precip_total_mm', 'tke_proxy', 'wind_speed_era5', 'wind_dir_era5', 'temp_2m_era5', 'rh_2m_era5', 'pressure_era5']


,date,blh_mean,blh_min,blh_max,blh_morning_mean,radiation_mean,precip_total_mm,tke_proxy,wind_speed_era5,wind_dir_era5,temp_2m_era5,rh_2m_era5,pressure_era5
count,2923,2923.000000,2923.000000,2923.000000,2922.000000,2923.000000,2923.000000,2923.000000,2923.000000,2923.000000,2923.000000,2923.000000,2923.000000
mean,2018-12-31 00:00:00,365.010895,21.400640,1377.567871,42.178143,217.720016,7.126064,0.156060,0.794053,117.254204,18.389011,80.804451,803.145569
min,2014-12-31 00:00:00,39.909733,11.699265,65.516083,12.556081,0.000000,0.000159,0.008299,0.271419,1.189667,15.239044,46.763020,799.797180
25%,2016-12-30 12:00:00,282.607605,16.897698,1119.544250,31.291897,187.561600,1.990954,0.103367,0.590874,90.399422,17.694824,76.397297,802.569763
50%,2018-12-31 00:00:00,346.510345,20.048576,1347.079224,38.872154,222.435684,5.698602,0.144823,0.742822,100.579178,18.342224,81.809204,803.163574
75%,2020-12-30 12:00:00,432.451843,24.642477,1622.095398,48.437935,251.906845,10.162632,0.193858,0.942357,124.418137,19.052811,86.221561,803.745911
max,2022-12-31 00:00:00,827.061462,109.555244,2702.456055,179.628418,330.150879,57.607334,0.676071,2.118626,355.320709,21.753784,96.538635,805.822266
std,NaN,116.077171,6.383404,395.036896,16.984882,47.160751,6.784097,0.073051,0.268029,44.894302,0.992037,7.463853,0.878348


## Rename ERA5-derived meteorology to standard column names

Medellín has no local gauge to prefer, so the `_era5` variants become the primary
`wind_speed`/`wind_dir`/`temp_2m`/`rh_2m`/`pressure` columns (matching the names used
throughout the Bogotá pipeline's engineered-feature code, so it can be reused as-is).

In [3]:
era5_daily = era5_daily.rename(columns={
    'wind_speed_era5': 'wind_speed',
    'wind_dir_era5': 'wind_dir',
    'temp_2m_era5': 'temp_2m',
    'rh_2m_era5': 'rh_2m',
    'pressure_era5': 'pressure',
})
out_path = OUT_DIR / "medellin_era5_daily.parquet"
era5_daily.to_parquet(out_path, index=False)
print("Saved:", out_path, era5_daily.shape)


Saved: /path/to/project/Data/processed/colombia/medellin/medellin_era5_daily.parquet (2923, 13)


## v6 extras (mirrors Bogotá's v6 pipeline): `tcwv`, `sensible_heat_flux`, `w700`, LTS

Previously missing for Medellín -- the pipeline had been treating ERA5's base 8-variable
pull as the full set. Downloaded now (see MASTER.md v10): `total_column_water_vapour`,
`surface_sensible_heat_flux` (both single-level, folded into `load_era5_daily`
automatically), plus pressure-level `vertical_velocity`@700hPa (`w700`) and
`temperature`@700hPa (for LTS, combined with the `t2m_lts`/`sp_lts` files already in the
base pull -- same `compute_lts_daily` used for Bogotá, self-consistent within one
reanalysis model).


In [4]:
import glob
from common.external_data import load_vertical_velocity_daily, compute_lts_daily

def _paths(pattern):
    return sorted(Path(p) for p in glob.glob(str(RAW_ERA5_DIR / pattern)))

tcwv_paths = _paths("bogota_era5_tcwv_*.nc")
sshf_paths = _paths("bogota_era5_sshf_*.nc")
w700_paths = _paths("bogota_era5_w700_*.nc")
t700_paths = _paths("bogota_era5_t700_*.nc")
t2m_lts_paths = _paths("bogota_era5_t2m_lts_*.nc")
sp_lts_paths = _paths("bogota_era5_sp_lts_*.nc")

print(f"tcwv: {len(tcwv_paths)}  sshf: {len(sshf_paths)}  w700: {len(w700_paths)}  "
      f"t700: {len(t700_paths)}  t2m_lts: {len(t2m_lts_paths)}  sp_lts: {len(sp_lts_paths)}")


tcwv: 8  sshf: 8  w700: 8  t700: 8  t2m_lts: 8  sp_lts: 8


In [5]:
# Recompute era5_daily including the v6 single-level extras (tcwv, sensible_heat_flux) --
# load_era5_daily merges whatever's present, so pass the base paths + the new ones.
all_singlelevel_paths = era5_paths + tcwv_paths + sshf_paths
era5_daily = load_era5_daily(all_singlelevel_paths)
era5_daily = era5_daily.rename(columns={
    'wind_speed_era5': 'wind_speed',
    'wind_dir_era5': 'wind_dir',
    'temp_2m_era5': 'temp_2m',
    'rh_2m_era5': 'rh_2m',
    'pressure_era5': 'pressure',
})
print(era5_daily.shape)
print(era5_daily[['tcwv', 'sensible_heat_flux', 'tke_proxy']].describe())

out_path = OUT_DIR / "medellin_era5_daily.parquet"
era5_daily.to_parquet(out_path, index=False)
print("Saved:", out_path, era5_daily.shape)


(2923, 15)
              tcwv  sensible_heat_flux    tke_proxy
count  2923.000000         2923.000000  2923.000000
mean     26.055101          -44.340595     0.156060
std       3.303399           16.933176     0.073051
min      10.198989          -97.358635     0.008299
25%      24.402842          -54.871803     0.103367
50%      26.636717          -43.219318     0.144823
75%      28.397736          -32.769049     0.193858
max      32.021587            3.010908     0.676071
Saved: /path/to/project/Data/processed/colombia/medellin/medellin_era5_daily.parquet (2923, 15)


In [6]:
w700_daily = load_vertical_velocity_daily(w700_paths)
print(w700_daily.shape)
print(w700_daily['w700'].describe())

w700_out_path = OUT_DIR / "medellin_w700_daily.parquet"
w700_daily.to_parquet(w700_out_path, index=False)
print("Saved:", w700_out_path, w700_daily.shape)


(2923, 2)
count    2923.000000
mean       -0.091747
std         0.095708
min        -0.480344
25%        -0.149622
50%        -0.095059
75%        -0.036161
max         0.477240
Name: w700, dtype: float64
Saved: /path/to/project/Data/processed/colombia/medellin/medellin_w700_daily.parquet (2923, 2)


In [7]:
lts_daily = compute_lts_daily(t2m_lts_paths, sp_lts_paths, t700_paths)
print(lts_daily.shape)
print(lts_daily['lts'].describe())

lts_out_path = OUT_DIR / "medellin_lts_daily.parquet"
lts_daily.to_parquet(lts_out_path, index=False)
print("Saved:", lts_out_path, lts_daily.shape)


(2923, 3)
count    2923.000000
mean        3.128907
std         0.637386
min         1.336632
25%         2.686627
50%         3.115927
75%         3.547083
max         5.869475
Name: lts, dtype: float64
Saved: /path/to/project/Data/processed/colombia/medellin/medellin_lts_daily.parquet (2923, 3)
